# Bearing Autoencoder — CWRU Dataset (todos los 17x)

**Dataset real:** Case Western Reserve University (CWRU) Bearing Dataset.
Se cargan **todos** los archivos `.mat` disponibles, clasificados por régimen.

**Arquitectura del experimento:**
- El AE se entrena **una sola vez** con señales sanas (Normal_0.mat → canal DE).
- Cada archivo de fallo es un **bearing independiente** con su propio `AdaptiveTau`.
- No hay datos sintéticos — todo es señal de campo real.

**Pasos:**
1. Imports & seeds
2. Carga del dataset CWRU — todos los `.mat`
3. Segmentación en ventanas
4. Entrenamiento del AE (solo señales sanas)
5. Verificación de reconstrucción
6. Distribuciones MSE por régimen
7. Calibración MC del umbral → z_MC
8. Clase AdaptiveTau
9. Experimento per-bearing (un tracker por archivo de fallo)
10. Sensibilidad al tamaño de ventana
11. Espacio latente PCA

## 0. Imports & seeds

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from pathlib import Path
from collections import deque

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)
tf.random.set_seed(42)
print('TF:', tf.__version__)

TF: 2.16.2


## 1. Carga del dataset CWRU

Estructura esperada (descarga estándar de CWRU):
```
data/CWRU/
├── Normal_0.mat      ← sano, carga 0 HP
├── Normal_1.mat      ← sano, carga 1 HP
├── Normal_2.mat      ← sano, carga 2 HP
├── Normal_3.mat      ← sano, carga 3 HP
├── B007_0.mat  B007_1.mat  B007_2.mat  B007_3.mat   ← ball fault 0.007"
├── B014_0.mat  ...                                   ← ball fault 0.014"
├── B021_0.mat  ...                                   ← ball fault 0.021"
├── IR007_0.mat ...                                   ← inner race 0.007"
├── IR014_0.mat ...                                   ← inner race 0.014"
├── IR021_0.mat ...                                   ← inner race 0.021"
├── OR007@6_0.mat ...                                 ← outer race 0.007" @6
├── OR014@6_0.mat ...                                 ← outer race 0.014" @6
└── OR021@6_0.mat ...                                 ← outer race 0.021" @6
```

El loader detecta automáticamente todos los `.mat` presentes y los clasifica.
Si tu carpeta tiene otro path, ajusta `CWRU_DIR` a continuación.

In [2]:
# ── Ajusta este path a donde tengas el dataset ──────────────────────────────
CWRU_DIR = Path('data/CWRU')

# ── Tamaño de ventana de segmentación ───────────────────────────────────────
SEG_LEN  = 1200   # muestras por segmento (igual que el simulador)
SEG_STEP = 600    # paso (50% overlap)


def load_mat_channel(mat_path: Path) -> np.ndarray:
    """
    Lee el canal Drive End (DE) de un archivo CWRU .mat.
    Devuelve array 1D de float64.
    """
    mat = sio.loadmat(str(mat_path))
    # Busca la clave que contiene 'DE_time' (Drive End)
    key = next(k for k in mat if 'DE_time' in k)
    return mat[key].squeeze().astype(np.float64)


def segment(signal: np.ndarray, length: int, step: int) -> np.ndarray:
    """Segmenta una señal en ventanas de 'length' con paso 'step'. Shape: (N, length)"""
    starts = range(0, len(signal) - length + 1, step)
    return np.stack([signal[s:s+length] for s in starts])


def classify_file(name: str):
    """
    Clasifica un nombre de archivo CWRU en (regime_key, label_human).
    Devuelve None si no reconoce el archivo.
    """
    n = name.upper()
    if n.startswith('NORMAL'):  return ('healthy',    'Healthy')
    if n.startswith('IR'):      return ('inner_race', f'Inner Race ({name})')
    if n.startswith('OR'):      return ('outer_race', f'Outer Race ({name})')
    if n.startswith('B'):       return ('ball_fault', f'Ball Fault ({name})')
    return None


# ── Cargar todos los .mat disponibles ───────────────────────────────────────
all_files = sorted(CWRU_DIR.glob('*.mat'))
print(f'Archivos .mat encontrados: {len(all_files)}')

raw_by_regime = {}   # regime_key → list of (filename, segments_array)
skipped = []

for fpath in all_files:
    cls = classify_file(fpath.stem)
    if cls is None:
        skipped.append(fpath.name)
        continue
    regime_key, human_label = cls
    try:
        sig  = load_mat_channel(fpath)
        segs = segment(sig, SEG_LEN, SEG_STEP)
        raw_by_regime.setdefault(regime_key, []).append(
            {'file': fpath.stem, 'label': human_label, 'segments': segs}
        )
        print(f'  [{regime_key:12s}]  {fpath.stem:20s}  → {segs.shape[0]} segmentos')
    except Exception as e:
        print(f'  ERROR leyendo {fpath.name}: {e}')

if skipped:
    print(f'Archivos omitidos (no reconocidos): {skipped}')

REGIMES_PRESENT = list(raw_by_regime.keys())
print(f'\nRegímenes presentes: {REGIMES_PRESENT}')

Archivos .mat encontrados: 0

Regímenes presentes: []


## 2. Construir dataset de entrenamiento y test

- **Train / Val:** todos los segmentos sanos (Normal_*.mat), split 80/20.
- **Test per-bearing:** cada archivo de fallo es un bearing independiente.
  Para cada uno construimos un stream realista:
  `N_healthy_seed segmentos sanos + todos los segmentos del fallo`.
  Los segmentos sanos del seed se toman de los archivos Normal_*.mat
  (distintos de los usados en train para evitar data leakage).

In [3]:
VAL_FRAC      = 0.2    # fracción del bloque sano para validación
N_HEALTHY_SEED = 100   # segmentos sanos que preceden a cada bearing de fallo

# ── Segmentos sanos ─────────────────────────────────────────────────────────
healthy_segs = np.concatenate(
    [e['segments'] for e in raw_by_regime.get('healthy', [])], axis=0
)
print(f'Total segmentos sanos: {len(healthy_segs)}')

n_val   = max(1, int(len(healthy_segs) * VAL_FRAC))
n_train = len(healthy_segs) - n_val

# Mezcla antes de dividir
rng = np.random.default_rng(42)
idx = rng.permutation(len(healthy_segs))
healthy_segs = healthy_segs[idx]

# Normalización — scaler ajustado solo en train
scaler = StandardScaler()
healthy_sc = scaler.fit_transform(healthy_segs)

X_train = healthy_sc[:n_train, :, np.newaxis].astype('float32')
X_val   = healthy_sc[n_train:, :, np.newaxis].astype('float32')
SIG_LEN = X_train.shape[1]

print(f'Train: {X_train.shape}   Val: {X_val.shape}   SIG_LEN={SIG_LEN}')

# ── Streams per-bearing (para el experimento de la sección 9) ────────────────
# Cada stream = N_HEALTHY_SEED segmentos sanos + todos los del fichero de fallo
bearing_streams = []   # list of dicts

# Pool de segmentos sanos para el seed (usamos los de val para no contaminar train)
healthy_seed_pool = healthy_sc[n_train:]  # mismos que val; cada bearing toma N_HEALTHY_SEED

FAULT_REGIMES = [r for r in REGIMES_PRESENT if r != 'healthy']
COLOR_MAP = {'inner_race': '#a12c7b', 'outer_race': '#da7101', 'ball_fault': '#006494'}

for regime in FAULT_REGIMES:
    for entry in raw_by_regime[regime]:
        seed_segs  = healthy_seed_pool[:N_HEALTHY_SEED]   # mismo pool para todos
        fault_segs = scaler.transform(entry['segments'])
        stream_raw = np.concatenate([seed_segs, fault_segs], axis=0)
        stream_X   = stream_raw[:, :, np.newaxis].astype('float32')

        bearing_streams.append({
            'file':       entry['file'],
            'label':      entry['label'],
            'regime':     regime,
            'color':      COLOR_MAP.get(regime, '#555555'),
            'n_healthy':  N_HEALTHY_SEED,
            'n_fault':    len(fault_segs),
            'stream_X':   stream_X,
        })

# También streams de control (solo sano — un bearing por fichero normal)
for entry in raw_by_regime.get('healthy', []):
    segs_sc = scaler.transform(entry['segments'])
    bearing_streams.append({
        'file':      entry['file'],
        'label':     f'Control sano ({entry["file"]})',
        'regime':    'healthy',
        'color':     '#01696f',
        'n_healthy': len(segs_sc),
        'n_fault':   0,
        'stream_X':  segs_sc[:, :, np.newaxis].astype('float32'),
    })

print(f'\nTotal bearing streams: {len(bearing_streams)}')
for bs in bearing_streams:
    print(f'  {bs["file"]:25s}  regime={bs["regime"]:12s}  '
          f'sano={bs["n_healthy"]}  fallo={bs["n_fault"]}')

ValueError: need at least one array to concatenate

## 3. Arquitectura del Autoencoder

Igual que en el notebook original — Conv1D encoder + Conv1DTranspose decoder.
Entrenado **una sola vez** con señales sanas reales.

In [ ]:
LATENT_DIM = 16

def build_autoencoder(sig_len, latent_dim):
    inp = keras.Input(shape=(sig_len, 1), name='signal_in')
    x   = layers.Conv1D(32,  16, strides=2, padding='same', activation='relu')(inp)
    x   = layers.Conv1D(64,   8, strides=2, padding='same', activation='relu')(x)
    x   = layers.Conv1D(128,  4, strides=2, padding='same', activation='relu')(x)
    conv_shape = x.shape[1:]
    x   = layers.Flatten()(x)
    latent = layers.Dense(
        latent_dim,
        activity_regularizer=keras.regularizers.l1(1.5e-4),
        name='latent'
    )(x)
    y = layers.Dense(conv_shape[0] * conv_shape[1], activation='relu')(latent)
    y = layers.Reshape(conv_shape)(y)
    y = layers.Conv1DTranspose(128, 4,  strides=2, padding='same', activation='relu')(y)
    y = layers.Conv1DTranspose(64,  8,  strides=2, padding='same', activation='relu')(y)
    y = layers.Conv1DTranspose(32,  16, strides=2, padding='same', activation='relu')(y)
    y = layers.Conv1D(1, 1, padding='same', activation='linear', name='signal_out')(y)
    if y.shape[1] > sig_len:
        y = layers.Cropping1D((0, y.shape[1] - sig_len))(y)
    ae      = Model(inp, y,      name='bearing_autoencoder')
    encoder = Model(inp, latent, name='encoder')
    return ae, encoder

ae, encoder = build_autoencoder(SIG_LEN, LATENT_DIM)
ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
ae.summary()

## 4. Entrenamiento (señales sanas reales — una sola vez)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1
    ),
]

history = ae.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=100, batch_size=32,
    callbacks=callbacks, verbose=1,
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history.history['loss'],     color='#01696f', label='train')
ax.plot(history.history['val_loss'], color='#a12c7b', ls='--', label='val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
ax.set_title('Curva de entrenamiento — AE sobre señales sanas CWRU')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Verificación de reconstrucción (val sano)

In [ ]:
def per_sample_mse(model, X):
    Xr = model.predict(X, verbose=0)
    return np.mean((X[:, :, 0] - Xr[:, :, 0]) ** 2, axis=1)

idx   = np.random.choice(len(X_val), 3, replace=False)
preds = ae.predict(X_val[idx], verbose=0)

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for k, ax in enumerate(axes):
    ax.plot(X_val[idx[k], :, 0], color='#01696f', lw=0.8, label='original')
    ax.plot(preds[k, :, 0],      color='#da7101', lw=0.8, ls='--', label='reconstruida')
    ax.set_ylabel('Amplitud'); ax.legend(fontsize=8)
axes[-1].set_xlabel('Muestra')
fig.suptitle('Señal sana CWRU — original vs reconstruida')
plt.tight_layout(); plt.show()

## 6. Distribuciones de MSE por régimen (todos los archivos CWRU)

Verificación de separabilidad: las señales sanas deben tener MSE bajo;
las señales de fallo deben desviarse según la severidad.

In [ ]:
REGIME_COLORS = {
    'healthy':    '#01696f',
    'inner_race': '#a12c7b',
    'outer_race': '#da7101',
    'ball_fault': '#006494',
}

fig, ax = plt.subplots(figsize=(14, 5))
xticks, xlabels = [], []
pos = 0

for regime in (['healthy'] + FAULT_REGIMES):
    entries = raw_by_regime.get(regime, [])
    color   = REGIME_COLORS.get(regime, '#555')
    for entry in entries:
        segs_sc = scaler.transform(entry['segments'])[:, :, np.newaxis].astype('float32')
        mse_arr = per_sample_mse(ae, segs_sc)
        ax.scatter(np.full_like(mse_arr, pos) + np.random.uniform(-0.2, 0.2, len(mse_arr)),
                   mse_arr, s=4, alpha=0.3, color=color)
        ax.boxplot(mse_arr, positions=[pos], widths=0.4,
                   boxprops=dict(color=color),
                   whiskerprops=dict(color=color),
                   capprops=dict(color=color),
                   medianprops=dict(color='white', lw=2),
                   flierprops=dict(marker=''))
        xticks.append(pos); xlabels.append(entry['file'])
        pos += 1
    pos += 0.5  # separador entre regímenes

ax.set_xticks(xticks); ax.set_xticklabels(xlabels, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('MSE reconstrucción')
ax.set_title('Anomaly score (MSE) por archivo CWRU — AE entrenado solo en sanos')
plt.tight_layout(); plt.show()

## 7. Calibración del umbral → z_MC

Se calcula sobre el conjunto de validación sano real (nunca visto en train).

$$z_{\text{MC}} = \frac{\tau_{\text{MC}} - \mu_{\text{MC}}}{\sigma_{\text{MC}}}$$

In [ ]:
mse_mc = per_sample_mse(ae, X_val)

mu_mc  = mse_mc.mean()
std_mc = mse_mc.std()
PERCENTILE = 99
tau_mc = np.percentile(mse_mc, PERCENTILE)
z_mc   = (tau_mc - mu_mc) / std_mc

print(f'Val sano CWRU   μ={mu_mc:.5f}  σ={std_mc:.5f}')
print(f'τ_MC ({PERCENTILE}th pct) = {tau_mc:.5f}   →  z_MC = {z_mc:.2f}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(mse_mc, bins=40, color='#01696f', alpha=0.7, label='Val sano CWRU')
ax.axvline(tau_mc, color='#a12c7b', lw=2, label=f'τ_MC ({PERCENTILE}th) = {tau_mc:.5f}')
ax.set_xlabel('MSE'); ax.set_ylabel('Frecuencia')
ax.set_title('Distribución MSE — calibración sobre señales sanas reales')
ax.legend(); plt.tight_layout(); plt.show()

## 8. AdaptiveTau — umbral dinámico por bearing

Un `AdaptiveTau` independiente por cada archivo de fallo.
El AE es compartido; el z_MC es compartido; la escala local la pone cada bearing.

```
τ_t = μ_t  +  z_MC · σ_t
```

El buffer se actualiza **solo** con muestras no anómalas → no hay contaminación.

In [ ]:
WINDOW_SIZE = 200   # ajustable — ver §10

class AdaptiveTau:
    def __init__(self, z_mc, window_size=200, warmup=50, tau_init=None):
        self.z_mc        = z_mc
        self.window_size = window_size
        self.warmup      = warmup
        self.tau_init    = tau_init
        self._buf        = deque(maxlen=window_size)

    def seed(self, mse_array):
        for v in mse_array[-self.window_size:]:
            self._buf.append(float(v))

    def update(self, mse_value):
        if len(self._buf) < max(2, self.warmup):
            tau   = self.tau_init if self.tau_init is not None else np.inf
            mu    = float(np.mean(list(self._buf))) if self._buf else 0.0
            sigma = float(np.std(list(self._buf)))  if len(self._buf) > 1 else 0.0
        else:
            arr   = np.array(self._buf)
            mu    = float(arr.mean())
            sigma = float(arr.std())
            tau   = mu + self.z_mc * sigma
        is_anomaly = bool(mse_value > tau)
        if not is_anomaly:
            self._buf.append(float(mse_value))
        return mu, sigma, tau, is_anomaly

print(f'AdaptiveTau listo  |  z_MC={z_mc:.2f}  window={WINDOW_SIZE}')

## 9. Experimento per-bearing — todos los archivos CWRU

Cada archivo de fallo del dataset es un bearing independiente.
El stream de cada uno: **100 segmentos sanos → todos los segmentos del fallo**.
Los bearings de control (Normal_*.mat) se evalúan completos como señales sanas.

Un AE compartido, un `AdaptiveTau` independiente por bearing.

In [ ]:
bearing_results = []

for bs in bearing_streams:
    stream_X   = bs['stream_X']
    stream_mse = per_sample_mse(ae, stream_X)

    tracker = AdaptiveTau(z_mc=z_mc, window_size=WINDOW_SIZE,
                          warmup=50, tau_init=tau_mc)
    tracker.seed(mse_mc)   # mismo warm-start para todos

    mus, sigmas, taus, flags = [], [], [], []
    for val in stream_mse:
        mu_t, sig_t, tau_t, flag = tracker.update(val)
        mus.append(mu_t); sigmas.append(sig_t)
        taus.append(tau_t); flags.append(flag)

    res = dict(
        **{k: bs[k] for k in ('file','label','regime','color','n_healthy','n_fault')},
        mse=stream_mse,
        taus=np.array(taus),
        mus=np.array(mus),
        sigmas=np.array(sigmas),
        flags=np.array(flags),
    )
    bearing_results.append(res)

# ── Métricas ────────────────────────────────────────────────────────────────
print(f'{"Archivo":<25} {"Régimen":<12} {"FAR":>8} {"DR":>8}')
print('-' * 60)
for r in bearing_results:
    nh = r['n_healthy']; nf = r['n_fault']
    far = r['flags'][:nh].sum() / nh if nh else float('nan')
    dr  = r['flags'][nh:].sum()  / nf if nf else float('nan')
    dr_s = f'{dr:.3f}' if nf else 'N/A (control)'
    print(f'{r["file"]:25s} {r["regime"]:12s} {far:8.3f} {dr_s:>8}')

In [ ]:
# ── Plot — un subplot por bearing ──────────────────────────────────────────
n_plots = len(bearing_results)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3.5 * n_plots), squeeze=False)

for ax, r in zip(axes[:, 0], bearing_results):
    t     = np.arange(len(r['mse']))
    flags = r['flags']
    nh    = r['n_healthy']

    # Fondo sano
    ax.axvspan(0, nh, color='#01696f22', zorder=0)
    ax.text(nh / 2, ax.get_ylim()[1], 'Sano', ha='center', fontsize=7, va='bottom')

    # Fondo fallo (si lo hay)
    if r['n_fault'] > 0:
        ax.axvspan(nh, len(t), color=r['color'] + '22', zorder=0)
        ax.text(nh + r['n_fault'] / 2, ax.get_ylim()[1],
                r['label'], ha='center', fontsize=7, va='bottom')

    ax.scatter(t[~flags], r['mse'][~flags], s=5,  color='#01696f', alpha=0.5, label='Normal')
    ax.scatter(t[ flags], r['mse'][ flags], s=20, color='red', marker='x', zorder=5, label='Alarma')
    ax.plot(t, r['taus'], color='#a12c7b', lw=1.2, label=f'τ_t (w={WINDOW_SIZE})')
    ax.axhline(tau_mc, color='gray', lw=1, ls='--', alpha=0.6, label='τ_MC estático')
    ax.fill_between(t, r['mus'] - r['sigmas'], r['mus'] + r['sigmas'],
                    color='#01696f', alpha=0.1)
    ax.set_title(f"{r['file']}  —  {r['regime']}", fontsize=9)
    ax.set_ylabel('MSE')
    ax.legend(fontsize=6, loc='upper left', ncol=4)

axes[-1, 0].set_xlabel('Índice de segmento')
fig.suptitle(
    f'Per-bearing AdaptiveTau — AE único — Dataset CWRU (todos los 17x)',
    fontsize=12, y=1.001
)
plt.tight_layout()
plt.savefig('cwru_per_bearing.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada en cwru_per_bearing.png')

## 10. Sensibilidad al tamaño de ventana

In [ ]:
window_sizes = [50, 100, 200, 500]
sweep = []

# Usar solo bearings con fallo para la curva
fault_streams = [bs for bs in bearing_streams if bs['regime'] != 'healthy']

for ws in window_sizes:
    far_list, dr_list = [], []
    for bs in fault_streams:
        mse = per_sample_mse(ae, bs['stream_X'])
        tr  = AdaptiveTau(z_mc=z_mc, window_size=ws, warmup=min(ws//4, 50), tau_init=tau_mc)
        tr.seed(mse_mc)
        flgs = np.array([tr.update(v)[3] for v in mse])
        nh = bs['n_healthy']; nf = bs['n_fault']
        far_list.append(flgs[:nh].sum() / nh)
        dr_list.append( flgs[nh:].sum() / nf)
    sweep.append((ws, np.mean(far_list), np.mean(dr_list)))
    print(f'window={ws:5d}  FAR={np.mean(far_list):.3f}  DR={np.mean(dr_list):.3f}')

sw = np.array(sweep)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sw[:,0], sw[:,1]*100, 'o--', color='#a12c7b', label='FAR sano (%)')
ax.plot(sw[:,0], sw[:,2]*100, 's-',  color='#01696f', label='Detection Rate (%)')
ax.set_xscale('log'); ax.set_xlabel('Window size'); ax.set_ylabel('%')
ax.set_title('FAR vs DR — sweep de WINDOW_SIZE sobre dataset CWRU')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Espacio latente — PCA

In [ ]:
segs_by_regime = {}
for regime in (['healthy'] + FAULT_REGIMES):
    entries = raw_by_regime.get(regime, [])
    if not entries: continue
    all_segs = np.concatenate([scaler.transform(e['segments']) for e in entries], axis=0)
    # cap at 500 for speed
    idx = np.random.choice(len(all_segs), min(500, len(all_segs)), replace=False)
    segs_by_regime[regime] = all_segs[idx]

X_all = np.concatenate(list(segs_by_regime.values()), axis=0)[:, :, np.newaxis].astype('float32')
y_all = np.concatenate([
    np.full(len(v), i) for i, v in enumerate(segs_by_regime.values())
])

Z_all = encoder.predict(X_all, verbose=0)
pca   = PCA(n_components=2)
Z2    = pca.fit_transform(Z_all)

regime_labels = list(segs_by_regime.keys())
colors_pca    = ['#01696f', '#a12c7b', '#da7101', '#006494']

fig, ax = plt.subplots(figsize=(8, 6))
for i, (regime, color) in enumerate(zip(regime_labels, colors_pca)):
    mask = y_all == i
    ax.scatter(Z2[mask,0], Z2[mask,1], s=10, alpha=0.5, color=color, label=regime)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Espacio latente — PCA (AE entrenado solo en sanos CWRU)')
ax.legend(markerscale=2); plt.tight_layout(); plt.show()